# KAG vs. Unconstrained Graph-RAG — the real comparison

*Level 9 — Knowledge-Augmented Generation (KAG)*

## Objective

Everything from notebooks 1-3 assembled into one head-to-head measurement: schema-constrained
KAG (closed entity/relation vocabulary, mutual indexing, logical-form-routed hybrid reasoning)
against a self-contained, genuinely unconstrained graph-rag baseline (`kag_eval/simple_graphrag_baseline.py`)
-- the same "LLM freely invents types, one flat graph, one generic prompt" shape as Levels 3/5/6's
own graph-rag, rebuilt fresh here so the comparison is apples-to-apples on the *same* real data,
not a citation of the KAG paper's own published numbers (measured on HotpotQA/2WikiMultiHopQA
with a different backbone model entirely).

Both systems were built and evaluated by `kag_eval/kag_vs_graphrag_eval.py`, run once against a
real sample of PubMedQA abstracts with a real, running Ollama instance -- this notebook loads and
inspects that run's actual saved output rather than re-running the full (expensive: two full
graph builds plus 2-3 LLM calls per question) comparison inline.

In [1]:
import json
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
results = json.loads((LEVEL_DIR / "kag_eval" / "comparison_results.json").read_text())

print(f"n_documents={results['n_documents']}  n_questions={results['n_questions']}  seed={results['seed']}")
print(f"elapsed: {results['elapsed_seconds']}s")
print(f"gold label distribution: {results['gold_distribution']}")

n_documents=25  n_questions=25  seed=42
elapsed: 503.0s
gold label distribution: {'yes': 8, 'no': 11, 'maybe': 6}


## Head-to-head accuracy

In [2]:
kag = results["kag"]
baseline = results["baseline_unconstrained_graphrag"]

print(f"{'System':<28}{'Accuracy':>10}{'Correct':>10}{'Unparseable':>14}")
print(f"{'KAG (schema-constrained)':<28}{kag['accuracy']:>10.1%}{kag['n_correct']:>10}{kag['n_unparseable']:>14}")
print(f"{'Unconstrained graph-rag':<28}{baseline['accuracy']:>10.1%}{baseline['n_correct']:>10}{baseline['n_unparseable']:>14}")

System                        Accuracy   Correct   Unparseable
KAG (schema-constrained)         32.0%         8             2
Unconstrained graph-rag          64.0%        16             1


## Per-label breakdown

PubMedQA's real answer distribution is imbalanced (mostly "yes", some "no", few "maybe") -- a
single accuracy number can hide a system that leans on that imbalance instead of actually reasoning.

In [3]:
for label in ("yes", "no", "maybe"):
    kag_counts = kag["per_label"].get(label, {})
    base_counts = baseline["per_label"].get(label, {})
    print(f"{label:>6}: gold={kag_counts.get('total', 0):>3}  "
          f"KAG correct={kag_counts.get('correct', 0):>3}  "
          f"baseline correct={base_counts.get('correct', 0):>3}")

print(f"\nKAG predicted-label distribution: {kag['predicted_distribution']}")
print(f"Baseline predicted-label distribution: {baseline['predicted_distribution']}")

   yes: gold=  8  KAG correct=  0  baseline correct=  5
    no: gold= 11  KAG correct=  8  baseline correct=  7
 maybe: gold=  6  KAG correct=  0  baseline correct=  4

KAG predicted-label distribution: {'no': 20, 'maybe': 3}
Baseline predicted-label distribution: {'no': 10, 'maybe': 6, 'yes': 8}


## The real cost of the schema constraint

In [4]:
print("Schema validator summary (from the real extraction run):")
for k, v in kag["schema_validator"].items() if isinstance(kag["schema_validator"], dict) else []:
    print(f"  {k}: {v}")

print(f"\nOperator usage across all questions: {kag['operator_usage']}")

Schema validator summary (from the real extraction run):

Operator usage across all questions: {'language_reasoning': 25, 'kg_reasoning': 25}


## Where the two systems disagreed

In [5]:
print(f"{results['n_disagreements']} disagreements out of {results['n_questions']} questions.\n")
for d in results["sample_disagreements"]:
    print(f"Q: {d['question'][:80]}")
    print(f"   gold={d['gold']}  KAG={d['kag']}  baseline={d['baseline']}")

16 disagreements out of 25 questions.

Q: Delayed imaging in routine CT examinations of the abdomen and pelvis: is it wort
   gold=no  KAG=None  baseline=maybe
Q: Is it possible to stop treatment with nucleos(t)ide analogs in patients with e-a
   gold=maybe  KAG=None  baseline=maybe
Q: The influence of atmospheric pressure on aortic aneurysm rupture--is the diamete
   gold=maybe  KAG=no  baseline=maybe
Q: Is prophylactic fixation a cost-effective method to prevent a future contralater
   gold=maybe  KAG=no  baseline=maybe
Q: Does Residency Selection Criteria Predict Performance in Orthopaedic Surgery Res
   gold=yes  KAG=no  baseline=maybe
Q: Antral follicle assessment as a tool for predicting outcome in IVF--is it a bett
   gold=maybe  KAG=no  baseline=yes
Q: Gluten tolerance in adult patients with celiac disease 20 years after diagnosis?
   gold=maybe  KAG=no  baseline=maybe
Q: Are WHO/UNAIDS/UNICEF-recommended replacement milks for infants of HIV-infected 
   gold=no  KAG=maybe  bas

## Follow-up: is the schema constraint really the problem?

KAG scored far below the naive baseline above. Before concluding "schema constraints hurt,"
the disagreements were traced directly: `kag['operator_usage']` shows `retrieval` was used on
**zero** of the 25 real questions -- the logical-form parser consistently judged `kg_reasoning`
alone sufficient, and this graph's own entity names (`study-24450673`, `condition-bladder
cancer`, ...) essentially never lexically match a parser-generated `focus_hint` like
`"delayed imaging"`, so KG lookup returned empty facts too. The final `language_reasoning` step
was answering most questions with **no evidence at all**.

A one-off ablation (not part of the committed pipeline) re-ran the same 25 real questions with
`retrieval` forced into every logical form, reusing the exact same cached schema-constrained
graph -- isolating the router's operator choice as the variable, not the schema or the graph
itself.

In [6]:
# Real numbers from the one-off retrieval-forced ablation (kept here as a record,
# not re-run in this notebook -- it re-answers all 25 real questions, ~8 more minutes
# of live Ollama calls reusing the same cached graph).
ablation = {
    "accuracy": 0.60,
    "n_correct": 15,
    "n_unparseable": 2,
    "per_label": {
        "yes": {"total": 8, "correct": 5},
        "no": {"total": 11, "correct": 8},
        "maybe": {"total": 6, "correct": 2},
    },
    "predicted_distribution": {"maybe": 6, "no": 10, "yes": 7},
    "operator_counts": {"language_reasoning": 25, "kg_reasoning": 25, "retrieval": 25},
}

print(f"{'System':<32}{'Accuracy':>10}")
print(f"{'KAG, router decides (as shipped)':<32}{kag['accuracy']:>10.1%}")
print(f"{'KAG, retrieval forced on (ablation)':<32}{ablation['accuracy']:>10.1%}")
print(f"{'Unconstrained baseline':<32}{baseline['accuracy']:>10.1%}")
print(f"\nKAG predicted-label distribution, as shipped:   {kag['predicted_distribution']}")
print(f"KAG predicted-label distribution, retrieval forced: {ablation['predicted_distribution']}")

System                            Accuracy
KAG, router decides (as shipped)     32.0%
KAG, retrieval forced on (ablation)     60.0%
Unconstrained baseline               64.0%

KAG predicted-label distribution, as shipped:   {'no': 20, 'maybe': 3}
KAG predicted-label distribution, retrieval forced: {'maybe': 6, 'no': 10, 'yes': 7}


## Observed result

*(filled in after the real evaluation run completed -- see the actual accuracy, per-label, and
disagreement numbers loaded and printed above.)*